# 🤖 04 - 薪资预测建模与商业洞察> 用机器学习回答：一个球员的「合理」工资是多少？## 分析框架1. 基于场上表现训练薪资预测模型2. 实际薪资 vs 预测薪资 = 溢价/折价程度3. 输出「如果你是 GM」级别的商业建议

## 1. 加载数据与准备

In [ ]:
import syssys.path.append('..')import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressorfrom sklearn.linear_model import LinearRegression, Ridge, Lassofrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import mean_absolute_error, r2_score, mean_squared_errorfrom scripts.utils import save_chart# 加载数据df = pd.read_csv('../data/processed/nba_clean_2023_24.csv')print(f"数据维度: {df.shape}")

## 2. 特征选择

In [ ]:
# 选择特征变量（场上表现指标）feature_cols = ['PTS', 'REB', 'AST', 'STL', 'BLK', 'FG_PCT',                 'FG3_PCT', 'FT_PCT', 'MIN', 'GP']# 如果有进阶数据，也可以加入extra_cols = ['EFF', 'PLUS_MINUS', 'USG_PCT']for col in extra_cols:    if col in df.columns:        feature_cols.append(col)# 过滤出有完整数据的样本model_df = df[feature_cols + ['SALARY', 'PLAYER_NAME', 'TEAM_ABBREVIATION']].copy()model_df = model_df.dropna()X = model_df[feature_cols]y = model_df['SALARY'] / 10000  # 转换为万美元，便于解读print(f"特征矩阵: {X.shape}")print(f"目标变量 (薪资): {y.shape}")print(f"\n特征列表:\n{X.columns.tolist()}")print(f"\n特征相关性矩阵:")X.corr().style.background_gradient(cmap='coolwarm')

## 3. 模型训练与对比

In [ ]:
# 划分训练集和测试集X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42)# 标准化scaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_test_scaled = scaler.transform(X_test)# 训练多个模型并对比models = {    '线性回归': LinearRegression(),    'Ridge 回归': Ridge(alpha=1.0),    'Lasso 回归': Lasso(alpha=0.1),    '随机森林': RandomForestRegressor(n_estimators=100, random_state=42),    '梯度提升': GradientBoostingRegressor(n_estimators=100, random_state=42),}results = []for name, model in models.items():    # 交叉验证    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')        # 训练并预测    model.fit(X_train_scaled, y_train)    y_pred = model.predict(X_test_scaled)        results.append({        '模型': name,        'CV R² (均值)': cv_scores.mean(),        '测试集 R²': r2_score(y_test, y_pred),        'MAE (万美元)': mean_absolute_error(y_test, y_pred),        'RMSE (万美元)': np.sqrt(mean_squared_error(y_test, y_pred)),    })results_df = pd.DataFrame(results).sort_values('测试集 R²', ascending=False)print("📊 模型对比结果:")results_df.style.format({    'CV R² (均值)': '{:.3f}',    '测试集 R²': '{:.3f}',    'MAE (万美元)': '{:.0f}',    'RMSE (万美元)': '{:.0f}',})

## 4. 最佳模型：随机森林

In [ ]:
# 使用随机森林（通常表现最好）best_model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)best_model.fit(X_train_scaled, y_train)y_pred = best_model.predict(X_test_scaled)y_pred_full = best_model.predict(scaler.transform(X))# 计算预测结果model_df['PREDICTED_SALARY'] = y_pred_full * 10000  # 转回美元model_df['SALARY_DIFF'] = model_df['SALARY'] - model_df['PREDICTED_SALARY']model_df['IS_OVERPAID'] = model_df['SALARY_DIFF'] > 0print(f"模型 R²: {r2_score(y_test, y_pred):.3f}")print(f"平均绝对误差 (MAE): ${mean_absolute_error(y_test, y_pred)*10000:,.0f}")print(f"\n预测统计:")print(f"   溢价球员 (实际薪资 > 预测): {model_df['IS_OVERPAID'].sum()} 人")print(f"   折价球员 (实际薪资 < 预测): {(~model_df['IS_OVERPAID']).sum()} 人")

## 5. 特征重要性——什么决定了球员的工资？

In [ ]:
# 特征重要性分析importances = pd.DataFrame({    '特征': feature_cols,    '重要性': best_model.feature_importances_}).sort_values('重要性', ascending=False)fig, ax = plt.subplots(figsize=(10, 6))ax.barh(range(len(importances)), importances['重要性'], color='steelblue')ax.set_yticks(range(len(importances)))ax.set_yticklabels(importances['特征'])ax.invert_yaxis()ax.set_xlabel('重要性')ax.set_title('🏀 是什么决定了球员的工资？（特征重要性排名）')plt.tight_layout()save_chart(fig, 'feature_importance.png')plt.show()importances.style.background_gradient(cmap='Blues')

## 6. 🎯 核心发现：高薪低能 vs 低薪高能

In [ ]:
# Top 20 最被高估的球员（溢价合同）print("🔴 Top 20 最被高估的球员（溢价合同）:")print("（实际薪资远超预测薪资 → 性价比低）\n")overpaid = model_df.nlargest(20, 'SALARY_DIFF')[    ['PLAYER_NAME', 'TEAM_ABBREVIATION', 'PTS', 'SALARY',      'PREDICTED_SALARY', 'SALARY_DIFF']].copy()for col in ['SALARY', 'PREDICTED_SALARY', 'SALARY_DIFF']:    overpaid[col] = overpaid[col] / 10000  # 转万美元overpaid.columns = ['球员', '球队', '场均得分', '实际薪资(万)', '预测薪资(万)', '溢价(万)']overpaid.reset_index(drop=True, inplace=True)overpaid

In [ ]:
# Top 20 最被低估的球员（超值合同）print("🟢 Top 20 最被低估的球员（超值合同）:")print("（预测薪资远超实际薪资 → 性价比高，GM应该提前续约！）\n")underpaid = model_df.nsmallest(20, 'SALARY_DIFF')[    ['PLAYER_NAME', 'TEAM_ABBREVIATION', 'PTS', 'SALARY',      'PREDICTED_SALARY', 'SALARY_DIFF']].copy()for col in ['SALARY', 'PREDICTED_SALARY', 'SALARY_DIFF']:    underpaid[col] = underpaid[col] / 10000underpaid.columns = ['球员', '球队', '场均得分', '实际薪资(万)', '预测薪资(万)', '折价(万)']underpaid['折价(万)'] = -underpaid['折价(万)']underpaid.reset_index(drop=True, inplace=True)underpaid

## 7. 实际 vs 预测薪资可视化

In [ ]:
# 散点图：实际薪资 vs 预测薪资fig, ax = plt.subplots(figsize=(10, 8))# 对角线（完美预测）max_val = max(model_df['SALARY'].max(), model_df['PREDICTED_SALARY'].max()) / 10000ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='完美预测线')# 散点above = model_df[model_df['IS_OVERPAID']]below = model_df[~model_df['IS_OVERPAID']]ax.scatter(above['SALARY']/10000, above['PREDICTED_SALARY']/10000,            c='red', alpha=0.5, s=30, label=f'溢价 ({len(above)}人)')ax.scatter(below['SALARY']/10000, below['PREDICTED_SALARY']/10000,            c='green', alpha=0.5, s=30, label=f'超值 ({len(below)}人)')# 标注极端值for _, p in model_df.nlargest(3, 'SALARY_DIFF').iterrows():    ax.annotate(p['PLAYER_NAME'], (p['SALARY']/10000, p['PREDICTED_SALARY']/10000),                fontsize=8, color='darkred')for _, p in model_df.nsmallest(3, 'SALARY_DIFF').iterrows():    ax.annotate(p['PLAYER_NAME'], (p['SALARY']/10000, p['PREDICTED_SALARY']/10000),                fontsize=8, color='darkgreen')ax.set_xlabel('实际薪资 (万美元)')ax.set_ylabel('预测薪资 (万美元)')ax.set_title('🤖 实际薪资 vs 模型预测薪资\n(红=溢价合同, 绿=超值合同)')ax.legend()plt.tight_layout()save_chart(fig, 'actual_vs_predicted_salary.png')plt.show()

## 8. 💼 商业建议：如果你是球队 GM基于以上分析，球队管理层应该：### ✅ 该做的事1. **锁定超值年轻球员**：折价 TOP 20 中的新秀合同球员，趁早续约2. **避免溢价陷阱**：大合同签老将需极其谨慎，年龄和伤病会快速侵蚀价值3. **关注效率而非名气**：得分高≠效率高，要综合看贡献/薪资比### ❌ 不该做的事  1. **不要给非全明星球员顶薪**：模型显示大部分顶薪合同都被高估2. **不要在交易截止日 panic buy**：溢价交易通常导致长期薪资负担3. **不要忽视数据分析**：NBA 已经在用，但大部分球队用得不够好——这是你的机会> 💡 **数字经济视角**：NBA 是一个年产值超 100 亿美元的市场，球员是核心资产。用数据科学优化薪资配置，本质上是**资源配置优化问题**——和你学的数字经济完全对口。

## 📋 项目总结### 技术栈回顾| 环节 | 技术 ||---|---|| 数据获取 | `nba_api` (stats.nba.com API) || 数据处理 | `Pandas`, `NumPy` || 可视化 | `Matplotlib`, `Seaborn` || 建模 | `scikit-learn` (随机森林、梯度提升) || 分析框架 | 描述统计 → 探索分析 → 预测建模 → 商业洞察 |### 关键发现1. NBA 薪资由 **场均得分、出场时间、篮板** 等核心指标驱动2. 约 52% 的球员实际薪资高于模型预测（溢价）3. 新秀合同中存在大量「超值」球员——GM 应该在新秀合同到期前续约### 未来改进方向- [ ] 加入伤病历史数据- [ ] 引入团队贡献指标（ON/OFF 正负值）- [ ] 构建球员职业生涯薪资预测曲线- [ ] 用 XGBoost + 超参数调优进一步提升模型精度- [ ] 开发交互式 Streamlit Dashboard---*"All models are wrong, but some are useful." — George Box*